# Hyper-parameter search (logged results)

The previous version of this notebook declared a 960-configuration x 4-fold MLP grid that was never run.
The search that was actually executed is **sequential one-factor sweeps selected on the validation split**
(`uv run python -m src tune --model all`, see `src/experiments/search.py`). This notebook only displays the
logs in `results/tuning/*.json`: every configuration that was evaluated, in order, with its cost.

In [ ]:
import sys
from pathlib import Path

# make `src` importable whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from IPython.display import Image, display

from src.experiments.search import CLASS_NAME, SPECS

tuning = {m: json.loads((ROOT / "results" / "tuning" / f"{CLASS_NAME[m]}.json").read_text()) for m in SPECS}
pd.DataFrame([{"model": CLASS_NAME[m], "configurations evaluated": t["n_trials"], "best validation accuracy": t["best_val_acc"],
               "selected": t["best_params"]} for m, t in tuning.items()]).set_index("model")

## Every evaluated configuration

In [ ]:
for m, t in tuning.items():
    print("\n==", CLASS_NAME[m], "==")
    df = pd.DataFrame([{"stage": r["stage"], "value": r["value"], "train_acc": r["train_acc"], "val_acc": r["val_acc"], "fit_s": r["train_time_sec"]}
                       for r in t["trials"]])
    display(df.style.format({"train_acc": "{:.4f}", "val_acc": "{:.4f}", "fit_s": "{:.1f}"}))

## One-factor sweep plots (train vs validation)

In [ ]:
for m in SPECS:
    display(Image(str(ROOT / "results" / "figures" / f"sweep_{m}.png")))

## Learning curves of the final models

In [ ]:
display(Image(str(ROOT / "results" / "figures" / "learning_curves.png")))